In [3]:
import Pkg; Pkg.add("StatsBase")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed StatsBase ─ v0.34.8
    Updating `~/.julia/environments/v1.10/Project.toml`
  [2913bbd2] + StatsBase v0.34.8
    Updating `~/.julia/environments/v1.10/Manifest.toml`
  [2913bbd2] ↑ StatsBase v0.34.7 ⇒ v0.34.8
Precompiling project...
  ✓ StatsBase
  1 dependency successfully precompiled in 4 seconds. 445 already precompiled.
  1 dependency precompiled but a different version is currently loaded. Restart julia to access the new version


In [4]:
using CSV
using DataFrames
using Flux
using Dates
using Statistics
using StatsBase # Per calcolare la frequenza degli IP
using Random

# 1. INCLUDIAMO IL TUO FILE DI UTILS
# ============================================================================
include("utils.jl") 

# 2. CARICAMENTO E PREPARAZIONE DATI
# ============================================================================
println("Caricamento dataset...")

# Sostituisci con il percorso reale del tuo file scaricato da Kaggle
# Supponiamo colonne tipiche: "Transaction Date", "Transaction Amount", "IP Address", "Is Fraud?"
df = CSV.read("Fraudulent_E-Commerce_Transaction_Data.csv", DataFrame)

# --- A. Gestione "Orari Particolari" ---
# Estraiamo l'ora dal timestamp. Le frodi avvengono spesso di notte.
# Assumiamo che la colonna data sia una stringa. Adattare il formato se necessario.
function get_hour(dt_str)
    try
        # Esempio formato: "2023-01-15 14:30:00"
        return hour(DateTime(dt_str, "yyyy-mm-dd HH:MM:SS")) 
    catch
        return 0 # Fallback
    end
end

# Creiamo la feature numerica per l'orario
# (Se il nome colonna è diverso nel CSV, cambialo qui sotto, es: :Date o :Timestamp)
df.Hour = get_hour.(df."Transaction Date")

# --- B. Rilevamento "IP Strani" (Comportamento Insolito) ---
# Una rete neurale non legge stringhe IP (es. "192.168.1.1").
# Usiamo la "Frequency Encoding": 
# - IP che appare 1 volta = Utente raro o nuovo (potenzialmente normale o usa e getta)
# - IP che appare 1000 volte in poco tempo = Bot/Comportamento insolito
ip_counts = countmap(df."IP Address")
df.IP_Frequency = [ip_counts[ip] for ip in df."IP Address"]

# --- C. Selezione Feature e Target ---
# Selezioniamo le colonne numeriche rilevanti per il training
input_cols = ["Transaction Amount", "Hour", "IP_Frequency"]
target_col = "Is Fraud?" # O il nome esatto della colonna target nel CSV

# Conversione in Matrice per Flux (Float32 è richiesto da Flux)
inputs = Matrix{Float32}(df[:, input_cols])

# Il target deve essere un vettore Booleano per le funzioni di utils.jl [cite: 5, 31]
# Assumiamo che nel CSV 1 = Frode, 0 = Legittimo
targets = df[:, target_col] .== 1

# Verifica dimensioni
println("Input size: ", size(inputs))
println("Target size: ", size(targets))

# 3. CONFIGURAZIONE RETE NEURALE E CROSS-VALIDATION
# ============================================================================

# Definiamo la topologia della rete:
# Input -> 16 neuroni nascosti -> 8 neuroni nascosti -> Output
topology = [16, 8] 

# Parametri di training
learning_rate = 0.01
max_epochs = 1000

# Creiamo gli indici per la k-fold cross-validation usando la tua funzione utils
k_folds = 5
# Usiamo la versione di crossvalidation per vettori Bool [cite: 32]
cv_indices = crossvalidation(targets, k_folds)

println("\nAvvio Cross-Validation ($k_folds folds)...")

# Chiamiamo la funzione principale dal tuo file utils.jl
# Questa funzione gestisce internamente la normalizzazione MinMax [cite: 41]
# e il calcolo delle metriche [cite: 50-54]
results = ANNCrossValidation(
    topology, 
    (inputs, targets), 
    cv_indices;
    maxEpochs=max_epochs, 
    learningRate=learning_rate,
    numExecutions=1, # Una esecuzione per fold per velocità
    print_results=true # Se la tua utils stampa, altrimenti stampiamo noi dopo
)

# 4. ANALISI DEI RISULTATI E CLASSIFICAZIONE RISCHIO
# ============================================================================

# Estrazione metriche medie restituite da ANNCrossValidation [cite: 54]
(meanAcc, stdAcc), (meanErr, stdErr), (meanSens, stdSens), 
(meanSpec, stdSpec), (meanPPV, stdPPV), _, (meanF1, stdF1), confMatrix = results

println("\n=== RISULTATI TRAINING ===")
println("Accuratezza Media: $(round(meanAcc * 100, digits=2))% ± $(round(stdAcc*100, digits=2))")
println("Sensibilità (Capacità di trovare frodi): $(round(meanSens * 100, digits=2))%")
println("Specificità (Capacità di ignorare onesti): $(round(meanSpec * 100, digits=2))%")
println("F1 Score: $(round(meanF1, digits=2))")

println("\nMatrice di Confusione Globale:")
display(confMatrix)

# 5. FUNZIONE DI UTILIZZO (INFERENZA)
# ============================================================================
# Questa funzione simula come usare il modello per classificare il rischio
# come richiesto: Nulla, Bassa, Media, Alta.

function classifica_rischio_frode(probabilita_frode::Float64)
    # La rete restituisce un valore tra 0 e 1 grazie alla sigmoide usata in buildClassANN [cite: 20]
    if probabilita_frode < 0.1
        return "Probabilità NULLA"
    elseif probabilita_frode < 0.4
        return "Probabilità BASSA"
    elseif probabilita_frode < 0.75
        return "Probabilità MEDIA (Check Richiesto)"
    else
        return "Probabilità ALTA (Bloccare Transazione)"
    end
end

println("\n--- Esempio di Classificazione su nuovi dati ---")
# Simuliamo 5 output casuali del modello (valori tra 0 e 1)
simulated_outputs = rand(5) 

for p in simulated_outputs
    livello = classifica_rischio_frode(p)
    println("Score Modello: $(round(p, digits=3)) -> Livello Rischio: $livello")
end

Caricamento dataset...


LoadError: ArgumentError: "Fraudulent_E-Commerce_Transaction_Data.csv" is not a valid file or doesn't exist